# 🛡️ SmartAPD - PPE Detection Model Training

Training YOLOv8 untuk deteksi PPE (Personal Protective Equipment)

**Dataset**: Construction Site Safety (Roboflow)

**Classes (10)**:
- ✅ Hardhat, Mask, Safety Vest (Safe)
- ❌ NO-Hardhat, NO-Mask, NO-Safety Vest (Violations)
- 🚧 Person, Safety Cone, machinery, vehicle

---

## 🚀 Cara Pakai:
1. Klik **Runtime** → **Change runtime type** → **T4 GPU**
2. Klik **Runtime** → **Run all**
3. Tunggu ~30-60 menit
4. Download model dari output

In [ ]:
# Check GPU
!nvidia-smi -L
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# Install dependencies
!pip install ultralytics==8.1.29 roboflow -qq

import ultralytics
print(f"Ultralytics version: {ultralytics.__version__}")

In [ ]:
# Imports
import os
import yaml
import warnings
from pathlib import Path
from ultralytics import YOLO

warnings.filterwarnings("ignore")

# Disable wandb
os.environ['WANDB_DISABLED'] = 'true'

## 📥 Download Dataset

In [ ]:
# Download dataset from Roboflow
from roboflow import Roboflow

# Public dataset - no API key needed
rf = Roboflow(api_key="")
project = rf.workspace("roboflow-universe-projects").project("construction-site-safety")
version = project.version(30)
dataset = version.download("yolov8")

print(f"\n✅ Dataset downloaded to: {dataset.location}")

In [ ]:
# Alternative: Download from Kaggle (if Roboflow fails)
# Uncomment if needed:

# !pip install kagglehub -qq
# import kagglehub
# path = kagglehub.dataset_download("snehilsanyal/construction-site-safety-image-dataset-roboflow")
# print(f"Dataset path: {path}")

## ⚙️ Configuration

In [ ]:
# Training Configuration
class CFG:
    SEED = 88
    
    # Classes
    CLASSES = [
        'Hardhat', 'Mask', 'NO-Hardhat', 'NO-Mask',
        'NO-Safety Vest', 'Person', 'Safety Cone',
        'Safety Vest', 'machinery', 'vehicle'
    ]
    NUM_CLASSES = len(CLASSES)
    
    # Training
    EPOCHS = 80
    BATCH_SIZE = 16
    IMG_SIZE = 640
    
    # Model - yolov8s is balanced (speed vs accuracy)
    BASE_MODEL = 'yolov8s.pt'
    
    # Optimizer
    OPTIMIZER = 'auto'
    LR = 1e-3
    LR_FACTOR = 0.01
    WEIGHT_DECAY = 5e-4
    DROPOUT = 0.025
    PATIENCE = 25
    
print(f"Training config:")
print(f"  - Epochs: {CFG.EPOCHS}")
print(f"  - Batch size: {CFG.BATCH_SIZE}")
print(f"  - Model: {CFG.BASE_MODEL}")
print(f"  - Classes: {CFG.NUM_CLASSES}")

## 🚀 Train Model

In [ ]:
# Load pretrained model
model = YOLO(CFG.BASE_MODEL)

# Find data.yaml path
data_yaml = f"{dataset.location}/data.yaml"
print(f"Data YAML: {data_yaml}")

# Show data.yaml content
with open(data_yaml, 'r') as f:
    print(f.read())

In [ ]:
%%time

# Train!
results = model.train(
    data=data_yaml,
    task='detect',
    
    epochs=CFG.EPOCHS,
    batch=CFG.BATCH_SIZE,
    imgsz=CFG.IMG_SIZE,
    
    optimizer=CFG.OPTIMIZER,
    lr0=CFG.LR,
    lrf=CFG.LR_FACTOR,
    weight_decay=CFG.WEIGHT_DECAY,
    dropout=CFG.DROPOUT,
    patience=CFG.PATIENCE,
    
    name='smartapd_ppe',
    seed=CFG.SEED,
    
    val=True,
    amp=True,
    exist_ok=True,
    verbose=True,
    device=0,
)

print("\n✅ Training complete!")

## 📊 Results

In [ ]:
# Show training results
from IPython.display import Image, display
import glob

results_dir = 'runs/detect/smartapd_ppe'

# Display metrics
for img_path in glob.glob(f'{results_dir}/*.png'):
    if 'batch' not in img_path:
        print(img_path)
        display(Image(filename=img_path, width=800))

## 💾 Export Model

In [ ]:
# Copy best model to easy location
import shutil

best_model = f'{results_dir}/weights/best.pt'
output_model = '/content/smartapd_ppe_model.pt'

shutil.copy(best_model, output_model)
print(f"✅ Model saved to: {output_model}")

# Model size
import os
size_mb = os.path.getsize(output_model) / (1024 * 1024)
print(f"📦 Model size: {size_mb:.2f} MB")

In [ ]:
# Export to ONNX (optional)
try:
    model.export(format='onnx', imgsz=CFG.IMG_SIZE)
    print("✅ ONNX export complete")
except Exception as e:
    print(f"⚠️ ONNX export failed: {e}")

## 📥 Download Model

Klik icon folder di sidebar kiri → Navigate ke `/content/smartapd_ppe_model.pt` → Right-click → Download

Atau run cell di bawah ini:

In [ ]:
# Download model file
from google.colab import files
files.download('/content/smartapd_ppe_model.pt')

## ✅ Done!

**Next Steps:**
1. Download file `smartapd_ppe_model.pt`
2. Pindahkan ke folder `ai-engine/` di project SmartAPD
3. Jalankan AI engine: `python web_server.py`

Model akan auto-load dan mulai deteksi PPE! 🎉